# Capítulo 7 · Recursos, prompts, sampling y elicitation

Lo que un servidor MCP puede ofrecer además de herramientas: documentos que la aplicación lee cuando los necesita (recursos), instrucciones preparadas (prompts), textos que el servidor pide al modelo del cliente (sampling) y preguntas que el servidor hace al usuario (elicitation). Se mide si los documentos conviene darlos todos o a demanda, y si preguntar al usuario evita las reservas con datos inventados.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo y la clínica

El servidor de la clínica es el del capítulo 6; se guarda en un archivo para importarlo.

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # las versiones con las que se comprobó el libro

In [ ]:
%%writefile servidor_clinica.py
"""Servidor MCP de la Clínica Veterinaria Naranco: el del capítulo 6, con las funciones de
las vacunas y de la reserva fuera del servidor, para reutilizarlas.

Se puede usar de tres formas: importarlo y conectar un cliente en memoria, lanzarlo como un
proceso que habla por stdio o lanzarlo como un servicio HTTP al que se conectan varios
clientes a la vez.
Uso: python servidor_clinica.py [carpeta_de_datos] [--http PUERTO]
"""
import argparse
import copy
import json
from datetime import date
from pathlib import Path
from typing import Annotated

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field

HOY = date(2026, 10, 20)                    # la fecha de los datos de la clínica


# --- Los datos de la clínica ----------------------------------------------------

def cargar_datos(carpeta="datos"):
    global MASCOTAS, DUENOS, HISTORIAL, VACUNAS, CITAS0, FARMACIA, FACTURAS, VETERINARIOS, HORAS
    leer = lambda archivo: json.loads((Path(carpeta) / archivo).read_text(encoding="utf-8"))
    MASCOTAS, DUENOS = leer("mascotas.json"), leer("duenos.json")
    HISTORIAL, VACUNAS = leer("historial.json"), leer("vacunas.json")
    CITAS0, FARMACIA = leer("citas.json"), leer("farmacia.json")
    FACTURAS, VETERINARIOS = leer("facturas.json"), leer("veterinarios.json")
    HORAS = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reiniciar()


def reiniciar():
    """Deja la clínica como al principio: la agenda original y ningún aviso."""
    global CITAS, AVISOS
    CITAS, AVISOS = copy.deepcopy(CITAS0), []


def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar nombres escritos de cualquier forma."""
    return str(texto).lower().translate(str.maketrans("áéíóúü", "aeiouu")).strip()


def edad(nacimiento):
    n = date.fromisoformat(nacimiento)
    return HOY.year - n.year - ((HOY.month, HOY.day) < (n.month, n.day))


def resumen_mascota(mid):
    m = MASCOTAS[mid]
    return {"nombre": m["nombre"], "especie": m["especie"], "raza": m["raza"],
            "edad_anos": edad(m["nacimiento"]), "peso_actual_kg": m["peso_kg"],
            "alergias": m["alergias"] or "ninguna conocida", "dueno": DUENOS[m["dueno"]]["nombre"]}


def veterinario_de(nombre):
    for v, n in VETERINARIOS.items():
        if sin_tildes(nombre) in (sin_tildes(v), sin_tildes(n)) or sin_tildes(nombre) in sin_tildes(n):
            return v
    return None


def mascota_de(nombre, dueno=""):
    ids = [m for m, d in MASCOTAS.items() if sin_tildes(nombre) == sin_tildes(d["nombre"])
           and sin_tildes(dueno) in sin_tildes(DUENOS[d["dueno"]]["nombre"])]
    if not ids:
        raise ToolError(f"No hay ninguna mascota llamada «{nombre}»"
                        + (f" de {dueno}" if dueno else "") + " en la clínica.")
    if len(ids) > 1:
        raise ToolError(f"Hay {len(ids)} mascotas llamadas «{nombre}»; indica el nombre del dueño.")
    return ids[0]


def libres(fecha, veterinario=""):
    try:
        dia = date.fromisoformat(fecha)
    except ValueError:
        raise ToolError(f"«{fecha}» no es una fecha; usa AAAA-MM-DD.")
    if dia.weekday() >= 5:
        raise ToolError(f"El {fecha} es fin de semana y la clínica cierra.")
    vet = veterinario_de(veterinario) if veterinario else None
    if veterinario and vet is None:
        raise ToolError(f"No hay ningún veterinario «{veterinario}». Son: "
                        + ", ".join(VETERINARIOS.values()) + ".")
    ocupadas = {(c["veterinario"], c["hora"]) for c in CITAS
                if c["fecha"] == fecha and c["estado"] != "cancelada"}
    return {"fecha": fecha, "horas_libres": {VETERINARIOS[v]: [h for h in HORAS if (v, h) not in ocupadas]
                                             for v in ([vet] if vet else VETERINARIOS)}}


def vacunas_de(mascota, dueno=""):
    """Las vacunas de una mascota, cada una con su próxima fecha y si está vencida."""
    mid = mascota_de(mascota, dueno)
    return {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(), "vacunas": [
        {"vacuna": v["vacuna"], "proxima": v["proxima"],
         "estado": "vencida" if v["proxima"] < HOY.isoformat() else "al día"} for v in VACUNAS[mid]]}


def reservar_cita_datos(mascota, veterinario, fecha, hora, motivo, dueno=""):
    """Reserva una cita, si todos los datos son válidos y la hora está libre."""
    mid, vet = mascota_de(mascota, dueno), veterinario_de(veterinario)
    if vet is None:
        raise ToolError(f"No hay ningún veterinario «{veterinario}».")
    libres(fecha, veterinario)
    if hora not in HORAS:
        raise ToolError(f"«{hora}» no es una hora de consulta.")
    if any(c["fecha"] == fecha and c["hora"] == hora and c["veterinario"] == vet
           and c["estado"] != "cancelada" for c in CITAS):
        raise ToolError(f"{VETERINARIOS[vet]} ya tiene una cita el {fecha} a las {hora}.")
    cita = {"id": f"CT-{len(CITAS0) + len(CITAS) + 1:04d}", "fecha": fecha, "hora": hora,
            "veterinario": vet, "mascota": mid, "motivo": motivo, "estado": "pendiente"}
    CITAS.append(cita)
    return {"reservada": cita["id"], "mascota": MASCOTAS[mid]["nombre"],
            "veterinario": VETERINARIOS[vet], "fecha": fecha, "hora": hora}


# --- El servidor ------------------------------------------------------------------

Nombre = Annotated[str, Field(description="Nombre de la mascota.")]
Dueno = Annotated[str, Field(description="Nombre del dueño; solo si hay varias mascotas con ese nombre.")]
Fecha = Annotated[str, Field(description="AAAA-MM-DD.")]


def crear_servidor():
    servidor = MCPServer("clinica", instructions="Mascotas, vacunas, historial y agenda de la Clínica "
                         "Veterinaria Naranco.")

    @servidor.tool()
    def buscar_mascota(nombre: Nombre, dueno: Dueno = "") -> dict:
        """Datos de una mascota buscada por su nombre: especie, edad, peso, alergias y dueño."""
        return resumen_mascota(mascota_de(nombre, dueno))

    @servidor.tool()
    def vacunas(mascota: Nombre, dueno: Dueno = "") -> dict:
        """Vacunas de una mascota: la próxima fecha de cada una y si está vencida."""
        return vacunas_de(mascota, dueno)

    @servidor.tool()
    def historial(mascota: Nombre, ultimas: Annotated[int, Field(
            description="Cuántas visitas, de la más reciente a la más antigua.")] = 3,
                  dueno: Dueno = "") -> dict:
        """Las últimas visitas de una mascota, empezando por la más reciente."""
        mid = mascota_de(mascota, dueno)
        return {"mascota": MASCOTAS[mid]["nombre"], "visitas": [
            {k: v[k] for k in ("fecha", "motivo", "tratamiento", "peso_kg")}
            for v in reversed(HISTORIAL[mid][-ultimas:])]}

    @servidor.tool()
    def huecos_libres(fecha: Fecha, veterinario: Annotated[str, Field(
            description="Nombre del veterinario; vacío para todos.")] = "") -> dict:
        """Horas libres de la agenda en un día laborable, de un veterinario o de todos."""
        return libres(fecha, veterinario)

    @servidor.tool()
    def citas_de(mascota: Nombre, dueno: Dueno = "") -> dict:
        """Citas de una mascota (pasadas y pendientes), con su id."""
        mid = mascota_de(mascota, dueno)
        return {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(), "citas": [
            {"id": c["id"], "fecha": c["fecha"], "hora": c["hora"], "motivo": c["motivo"],
             "veterinario": VETERINARIOS[c["veterinario"]], "estado": c["estado"]}
            for c in CITAS if c["mascota"] == mid]}

    @servidor.tool()
    def reservar_cita(mascota: Nombre,
                      veterinario: Annotated[str, Field(description="Nombre del veterinario.")],
                      fecha: Fecha, hora: Annotated[str, Field(description="HH:MM, en punto o y media.")],
                      motivo: Annotated[str, Field(description="Motivo de la cita.")],
                      dueno: Dueno = "") -> dict:
        """Reserva una cita en una hora libre."""
        return reservar_cita_datos(mascota, veterinario, fecha, hora, motivo, dueno)

    @servidor.tool()
    def cancelar_cita(cita: Annotated[str, Field(description="Id de la cita, como CT-0026 "
                                                             "(lo da citas_de).")]) -> dict:
        """Cancela una cita pendiente."""
        for c in CITAS:
            if c["id"] == cita:
                if c["estado"] != "pendiente":
                    raise ToolError(f"La cita {cita} está «{c['estado']}».")
                c["estado"] = "cancelada"
                return {"cancelada": cita, "fecha": c["fecha"], "hora": c["hora"]}
        raise ToolError(f"No existe la cita {cita}.")

    @servidor.tool()
    def citas_pendientes() -> dict:
        """Todas las citas pendientes de la agenda, para comprobar qué se ha reservado."""
        return {"citas": [{k: c[k] for k in ("id", "fecha", "hora", "veterinario", "mascota")}
                          for c in CITAS if c["estado"] == "pendiente"]}

    return servidor


if __name__ == "__main__":
    argumentos = argparse.ArgumentParser()
    argumentos.add_argument("datos", nargs="?", default="datos")
    argumentos.add_argument("--http", type=int, help="puerto; sin él, el servidor habla por stdio")
    a = argumentos.parse_args()
    cargar_datos(a.datos)
    servidor = crear_servidor()
    if a.http:
        servidor.run(transport="streamable-http", host="127.0.0.1", port=a.http)
    else:
        servidor.run()

In [ ]:
"""Capítulo 7: recursos, prompts, sampling y elicitation. Lo que un servidor MCP ofrece
además de herramientas, con dos medidas: documentos a demanda y preguntas al usuario."""
import asyncio
import json
import re
import sys
import urllib.request
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client
from mcp.server.mcpserver import Context
from mcp.server.mcpserver.exceptions import ToolError
from mcp.server.mcpserver.resources import TextResource
from mcp.types import CreateMessageResult, ElicitResult, SamplingMessage, TextContent
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
MODELO = None                               # None: elegir según el entorno
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/"
HOY = date(2026, 10, 20)                    # la fecha de los datos de la clínica


import servidor_clinica


def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def traer_datos():
    """Los archivos de la clínica, en la carpeta datos; en Colab, se descargan."""
    DATOS.mkdir(exist_ok=True)
    for archivo in ("mascotas", "duenos", "historial", "vacunas", "citas", "farmacia",
                    "facturas", "veterinarios", "tarifas"):
        ruta = DATOS / f"{archivo}.json"
        if not ruta.exists():
            urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)

In [ ]:
tok, modelo = cargar(MODELO)
traer_datos()
servidor_clinica.cargar_datos(DATOS)

In [ ]:
SISTEMA = (
    "You are the assistant of the staff of Clínica Veterinaria Naranco, a veterinary "
    "clinic in Oviedo (Spain). Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in Spanish, in one or two sentences.")


def generar(mensajes, esquemas, max_tokens=400):
    """Una respuesta del modelo y los tokens de entrada y de salida."""
    texto = tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False,
                                    add_generation_prompt=True)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    return (tok.decode(nuevos, skip_special_tokens=True).strip(),
            entrada["input_ids"].shape[1], len(nuevos))


def leer_llamadas(respuesta):
    """Las llamadas que pide el modelo, escritas entre <tool_call> y </tool_call>."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)</tool_call>", respuesta, re.S):
        try:
            llamada = json.loads(bloque)
            llamadas.append({"name": llamada["name"],
                             "arguments": llamada.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def dice(respuesta, *formas):
    """¿Aparece en la respuesta alguna de esas formas?"""
    return any(re.search(f, respuesta or "", re.I) for f in formas)


def sin_titulos(esquema):
    """El esquema sin las claves «title» que añade Pydantic y que el modelo no necesita."""
    if isinstance(esquema, dict):
        return {k: sin_titulos(v) for k, v in esquema.items() if k != "title"}
    return esquema


def compactar(texto):
    """El JSON de un resultado en una sola línea, sin la sangría con que lo envía el servidor."""
    try:
        return json.dumps(json.loads(texto), ensure_ascii=False)
    except ValueError:
        return texto

## 2. El servidor: herramientas, recursos, un prompt y sampling

In [ ]:
DOCUMENTOS = {
    "cancelaciones": ("Política de cancelación de citas",
        "Las citas se pueden cancelar o cambiar sin coste hasta 24 horas antes. Si se cancelan con "
        "menos de 24 horas de antelación, o el cliente no se presenta, se cobran 15 euros, que se "
        "descuentan de la siguiente consulta si se reserva en el mismo mes. Las cancelaciones de "
        "urgencia por enfermedad del animal no se cobran. Las cancelaciones se hacen por teléfono o "
        "en la recepción; no por correo electrónico."),
    "anestesia": ("Preparación para cirugía y anestesia",
        "Antes de una anestesia general, perros y gatos deben estar 12 horas sin comer; pueden beber "
        "agua hasta 2 horas antes. Los conejos no deben ayunar: se les retira la comida solo 1 hora "
        "antes. El día de la cirugía, el animal llega a las 8:30 y se recoge a partir de las 17:00. "
        "A los mayores de 8 años se les hace un análisis de sangre preoperatorio, que cuesta 65 euros."),
    "urgencias": ("Urgencias fuera de horario",
        "Fuera del horario de consulta, las urgencias las atiende el veterinario de guardia en el "
        "teléfono 985 000 111. La guardia rota cada semana entre los tres veterinarios. Una urgencia "
        "fuera de horario tiene un recargo de 60 euros sobre la tarifa normal. Los golpes de calor, "
        "las intoxicaciones y las dificultades para respirar son siempre urgencias."),
    "cachorros": ("Plan de vacunación de cachorros",
        "Los cachorros de perro reciben la primera dosis de la vacuna polivalente a las 8 semanas, la "
        "segunda a las 12 y la tercera a las 16, junto con la primera de la rabia. Hasta una semana "
        "después de la tercera dosis no deben salir a la calle ni relacionarse con otros perros. Los "
        "gatitos reciben la trivalente felina a las 9 y a las 12 semanas."),
    "desparasitacion": ("Calendario de desparasitación",
        "Los perros y gatos adultos se desparasitan por dentro cada 3 meses y por fuera cada mes, con "
        "pipeta o collar. Los cachorros, cada 15 días hasta los 3 meses de edad. Antes de una vacuna, "
        "el animal debe estar desparasitado por dentro en las 2 semanas anteriores; si no, se "
        "desparasita en la consulta y la vacuna se pone a los 10 días."),
    "pagos": ("Pagos y facturas",
        "Las consultas se pagan al terminar, con tarjeta, efectivo o Bizum. Las cirugías se pagan "
        "la mitad al reservar y el resto al recoger al animal. Las facturas sin pagar se recuerdan "
        "a los 30 días; a los 60 días no se dan nuevas citas, salvo urgencias, hasta que se paguen. "
        "Los clientes con más de tres mascotas tienen un 10 % de descuento en las vacunas."),
    "castracion": ("Cuidados después de una castración",
        "Tras una castración, el animal lleva un collar isabelino durante 10 días para que no se "
        "lama la herida. La revisión de puntos es a los 10 días y no se cobra. Durante una semana, "
        "paseos cortos y sin correr. Si la herida se abre, sangra o se hincha, hay que llamar a la "
        "clínica el mismo día."),
    "calor": ("Golpe de calor",
        "Ante un golpe de calor, hay que llevar al animal a la sombra, mojarlo con agua fresca, nunca "
        "helada, y acudir a la clínica de inmediato. Las razas chatas, como el bulldog francés o el "
        "persa, tienen más riesgo. En verano, los paseos deben ser antes de las 10:00 o después de "
        "las 20:00."),
}


MAS_DOCUMENTOS = {
    "microchip": ("Identificación con microchip",
        "Todos los perros, gatos y hurones deben llevar microchip desde los tres meses de edad. La "
        "clínica lo implanta en la consulta, sin anestesia, por 35 euros, y da de alta al animal en "
        "el registro autonómico en el mismo día. Si el dueño cambia de domicilio o de teléfono, debe "
        "comunicarlo a la clínica para actualizar el registro."),
    "viajes": ("Viajes y pasaporte",
        "Para viajar por la Unión Europea, el animal necesita microchip, pasaporte europeo y la vacuna "
        "de la rabia puesta al menos 21 días antes del viaje. El pasaporte cuesta 20 euros y se "
        "entrega en la consulta. Algunos países, como Irlanda o Finlandia, exigen además un "
        "tratamiento contra la tenia entre 1 y 5 días antes de llegar."),
    "residencia": ("Residencia y guardería",
        "La clínica no tiene residencia propia, pero colabora con la Residencia Canina del Naranco. "
        "Para ingresar en ella, el perro debe tener al día la polivalente, la rabia y la vacuna de la "
        "tos de las perreras, puesta al menos 15 días antes. La clínica expide el certificado sin "
        "coste a sus clientes."),
    "verano": ("Horario de verano",
        "En julio y agosto, la clínica abre de lunes a viernes de 9:00 a 14:00 y por la tarde solo "
        "los martes y los jueves, de 17:00 a 20:00. Las urgencias siguen atendidas por el veterinario "
        "de guardia. Las cirugías programadas se concentran en junio y septiembre."),
    "adopcion": ("Revisión de animales adoptados",
        "Los animales adoptados en protectoras de Asturias tienen la primera revisión gratis si se "
        "pide en el primer mes. En esa consulta se revisa el estado general, se comprueba el "
        "microchip y se planifican las vacunas y la desparasitación que falten."),
    "despedida": ("Final de la vida",
        "Cuando un animal está en sus últimos días, el veterinario explica a la familia las opciones "
        "de cuidados paliativos y, si se decide, la eutanasia, que se hace siempre con sedación "
        "previa. La clínica gestiona la incineración individual o colectiva con un servicio externo "
        "y no cobra la consulta de despedida."),
    "prospecto_meloxicam": ("Prospecto: meloxicam",
        "Antiinflamatorio para perros y gatos, en suspensión oral de 1,5 mg/ml. Dosis: 0,1 mg por kilo "
        "una vez al día, con comida. No dar a animales deshidratados ni con problemas de riñón. Si "
        "aparecen vómitos o heces negras, suspender y llamar a la clínica."),
    "prospecto_amoxicilina": ("Prospecto: amoxicilina",
        "Antibiótico para perros, gatos y conejos, en comprimidos de 50 mg. Dosis: 10 mg por kilo dos "
        "veces al día durante 7 días, aunque el animal mejore antes. No usar en animales alérgicos a "
        "las penicilinas. En conejos, vigilar las heces."),
    "prospecto_doxiciclina": ("Prospecto: doxiciclina",
        "Antibiótico para perros y gatos, en comprimidos de 100 mg. Dosis: 5 mg por kilo una vez al "
        "día. Dar con comida y con agua, nunca en seco, sobre todo a los gatos. No usar en cachorros "
        "en crecimiento."),
    "prospecto_maropitant": ("Prospecto: maropitant",
        "Antivomitivo solo para perros, en comprimidos de 16 mg. Dosis: 2 mg por kilo una vez al día, "
        "como máximo 5 días seguidos. Para el mareo en los viajes, darlo una hora antes de salir."),
    "prospecto_prazicuantel": ("Prospecto: prazicuantel",
        "Antiparasitario interno para perros y gatos, en comprimidos de 50 mg. Dosis: 5 mg por kilo en "
        "una sola toma. Es el tratamiento contra la tenia que exigen algunos países para viajar."),
    "prospecto_enrofloxacino": ("Prospecto: enrofloxacino",
        "Antibiótico para perros, gatos y conejos, en comprimidos de 15 mg. Dosis: 5 mg por kilo una "
        "vez al día. No usar en perros de menos de un año ni en gatos de menos de ocho semanas."),
    "prospecto_gabapentina": ("Prospecto: gabapentina",
        "Para el dolor crónico y la ansiedad antes de una visita, en perros y gatos, en cápsulas de "
        "100 mg. Dosis: 10 mg por kilo. Puede dar sueño; para las visitas, darla dos horas antes."),
}

In [ ]:
Nombre = Annotated[str, Field(description="Nombre de la mascota.")]


class FaltaHora(BaseModel):
    hora: str = Field(description="Hora de la cita, HH:MM.")


class FaltaVeterinario(BaseModel):
    veterinario: str = Field(description="Nombre del veterinario.")


class FaltaFecha(BaseModel):
    fecha: str = Field(description="Día de la cita, AAAA-MM-DD.")


class FaltaMotivo(BaseModel):
    motivo: str = Field(description="Motivo de la cita.")


FALTAN = {"hora": FaltaHora, "veterinario": FaltaVeterinario, "fecha": FaltaFecha, "motivo": FaltaMotivo}


class Confirmacion(BaseModel):
    veterinario: str = Field(description="Nombre del veterinario.")
    fecha: str = Field(description="Día de la cita, AAAA-MM-DD.")
    hora: str = Field(description="Hora de la cita, HH:MM.")
    motivo: str = Field(description="Motivo de la cita.")


def crear_servidor(preguntar="no", documentos=DOCUMENTOS):
    """El servidor de la clínica del capítulo 6, con los documentos como recursos, un prompt y
    una herramienta que redacta con el modelo del cliente. preguntar dice qué hace
    reservar_cita: «no» reserva lo que le pidan; «falta» acepta datos vacíos y pregunta al
    usuario los que faltan; «confirmar» enseña al usuario la cita antes de reservarla, para
    que corrija lo que no sea lo que pidió (las dos últimas, con elicitation)."""
    servidor = servidor_clinica.crear_servidor()
    servidor.remove_tool("reservar_cita")
    reservar = servidor_clinica.reservar_cita_datos

    if preguntar == "falta":
        @servidor.tool()
        async def reservar_cita(mascota: Nombre, ctx: Context, veterinario: str = "", fecha: str = "",
                                hora: str = "", motivo: str = "") -> dict:
            """Reserva una cita. Deja vacío cualquier dato que no te hayan dicho: se lo preguntará al usuario."""
            datos = {"veterinario": veterinario, "fecha": fecha, "hora": hora, "motivo": motivo}
            for campo, valor in datos.items():
                if not valor:
                    r = await ctx.elicit(f"Para reservar la cita de {mascota}, ¿{campo}?", FALTAN[campo])
                    if r.action != "accept":
                        raise ToolError(f"Reserva cancelada: falta {campo}.")
                    datos[campo] = getattr(r.data, campo)
            return reservar(mascota, **datos)
    elif preguntar == "confirmar":
        @servidor.tool()
        async def reservar_cita(mascota: Nombre, veterinario: str, fecha: str, hora: str, motivo: str,
                                ctx: Context) -> dict:
            """Reserva una cita, después de que el usuario la confirme. Fechas AAAA-MM-DD; horas HH:MM."""
            r = await ctx.elicit(f"Voy a reservar para {mascota}: {veterinario}, {fecha} a las {hora}, "
                                 f"motivo «{motivo}». Corrige lo que no sea correcto.", Confirmacion)
            if r.action != "accept":
                raise ToolError("El usuario no ha confirmado la reserva.")
            propuesta = {"veterinario": veterinario, "fecha": fecha, "hora": hora, "motivo": motivo}
            cambios = {c: f"{propuesta[c]} → {v}" for c, v in r.data.model_dump().items() if v != propuesta[c]}
            return {**reservar(mascota, **r.data.model_dump()),
                    "cambiado_por_el_usuario": cambios or "nada"}
    else:
        @servidor.tool()
        def reservar_cita(mascota: Nombre, veterinario: str, fecha: str, hora: str, motivo: str) -> dict:
            """Reserva una cita. Fechas AAAA-MM-DD; horas HH:MM, en punto o y media."""
            return reservar(mascota, veterinario=veterinario, fecha=fecha, hora=hora, motivo=motivo)

    for clave, (titulo, texto) in documentos.items():
        servidor.add_resource(TextResource(uri=f"clinica://documentos/{clave}", name=clave,
                                           description=titulo, text=texto))

    @servidor.prompt()
    def aviso_de_vacunas(mascota: str) -> str:
        """Pide al modelo un aviso de vacunas vencidas para el dueño de una mascota."""
        return (f"Consulta las vacunas de {mascota} con la herramienta vacunas y redacta, en dos "
                "frases y en un tono amable, un aviso para su dueño con las que están vencidas.")

    @servidor.tool()
    async def redactar_aviso(mascota: Nombre, ctx: Context) -> dict:
        """Redacta el texto de un aviso de vacunas vencidas para el dueño de una mascota."""
        datos = servidor_clinica.vacunas_de(mascota)
        vencidas = [v["vacuna"] for v in datos["vacunas"] if v["estado"] == "vencida"]
        peticion = (f"Escribe en una frase un aviso amable para el dueño de {mascota}: tiene "
                    f"vencidas estas vacunas: {', '.join(vencidas) or 'ninguna'}.")
        r = await ctx.session.create_message(
            messages=[SamplingMessage(role="user", content=TextContent(type="text", text=peticion))],
            max_tokens=80)
        return {"mascota": mascota, "vencidas": vencidas, "aviso": r.content.text}

    return servidor

## 3. El anfitrión

Da los documentos al modelo de dos formas, contesta las preguntas del servidor con un usuario simulado y atiende las peticiones de sampling con el modelo local.

In [ ]:
class Anfitrion:
    """El anfitrión compacto del capítulo 4, para un servidor, con tres añadidos: puede dar al
    modelo los documentos de dos formas (todos en el mensaje de sistema o una herramienta para
    leerlos cuando haga falta), contesta las preguntas del servidor con un usuario simulado y
    atiende las peticiones de sampling con el modelo local."""

    def __init__(self, servidor, documentos="ninguno", usuario=None):
        self.servidor, self.documentos, self.usuario = servidor, documentos, usuario
        self.preguntas = []

    async def __aenter__(self):
        self.cliente = Client(self.servidor, mode="legacy", elicitation_callback=self.responder,
                              sampling_callback=self.muestrear)
        await self.cliente.__aenter__()
        self.esquemas = [{"type": "function", "function": {
            "name": h.name, "description": h.description, "parameters": sin_titulos(h.input_schema)}}
            for h in (await self.cliente.list_tools()).tools if h.name != "redactar_aviso"]
        recursos = (await self.cliente.list_resources()).resources
        self.extra = ""
        if self.documentos == "todos":
            textos = [(r.description, (await self.cliente.read_resource(r.uri)).contents[0].text)
                      for r in recursos if str(r.uri).startswith("clinica://documentos/")]
            self.extra = "\n\nDocuments of the clinic:\n" + "\n".join(f"# {t}\n{x}" for t, x in textos)
        elif self.documentos == "a demanda":
            uris = [str(r.uri) for r in recursos if str(r.uri).startswith("clinica://documentos/")]
            self.esquemas.append({"type": "function", "function": {
                "name": "leer_documento", "description": "Lee un documento de la clínica. Documentos: "
                + "; ".join(f"{str(r.uri)} ({r.description})" for r in recursos if str(r.uri) in uris) + ".",
                "parameters": {"type": "object", "required": ["uri"],
                               "properties": {"uri": {"type": "string", "enum": uris}}}}})
        return self

    async def __aexit__(self, *error):
        await self.cliente.__aexit__(*error)

    async def responder(self, contexto, params):
        """Elicitation: el usuario simulado, que sabe qué cita quiere, contesta lo que le
        pregunte el servidor con los datos de esa cita."""
        campos = list(params.requested_schema["properties"])
        self.preguntas.append(params.message)
        if self.usuario and all(c in self.usuario for c in campos):
            return ElicitResult(action="accept", content={c: self.usuario[c] for c in campos})
        return ElicitResult(action="decline")

    async def muestrear(self, contexto, params):
        """Sampling: el servidor pide un texto y lo escribe el modelo del anfitrión."""
        mensajes = [{"role": m.role, "content": m.content.text} for m in params.messages]
        texto = generar(mensajes, [], max_tokens=params.max_tokens)[0]
        return CreateMessageResult(role="assistant", content=TextContent(type="text", text=texto),
                                   model=MODELO_GPU)

    async def llamar(self, nombre, argumentos):
        if nombre == "leer_documento":
            r = await self.cliente.read_resource(argumentos.get("uri", ""))
            return r.contents[0].text
        r = await self.cliente.call_tool(nombre, argumentos)
        return compactar("\n".join(b.text for b in r.content if b.type == "text"))


async def agente_mcp(tarea, anfitrion, max_pasos=8, ver=False):
    """El bucle del capítulo 4, con el texto extra del anfitrión en el mensaje de sistema."""
    mensajes = [{"role": "system", "content": SISTEMA + anfitrion.extra},
                {"role": "user", "content": tarea}]
    traza = []
    for paso in range(1, max_pasos + 1):
        respuesta, entrada, salida = generar(mensajes, anfitrion.esquemas)
        llamadas = leer_llamadas(respuesta)[:1]
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        traza.append({"paso": paso, "tokens_entrada": entrada, "llamadas": []})
        if not llamadas:
            if ver:
                print(f"[{paso}] Respuesta: {respuesta}")
            return respuesta, traza
        mensajes.append({"role": "assistant", "content": pensamiento, "tool_calls": [
            {"type": "function", "function": l} for l in llamadas if l["name"]]})
        for llamada in llamadas:
            texto = await anfitrion.llamar(llamada["name"], llamada["arguments"])
            traza[-1]["llamadas"].append({**llamada, "resultado": texto})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?", "content": texto})
            if ver:
                print(f"[{paso}] {llamada['name']}({json.dumps(llamada['arguments'], ensure_ascii=False)})")
                print(f"     → {texto[:150]}")
    return None, traza

In [ ]:
async def demostracion():
    async with Anfitrion(crear_servidor()) as a:
        recursos = (await a.cliente.list_resources()).resources
        print([str(r.uri) for r in recursos])
        prompt = await a.cliente.get_prompt("aviso_de_vacunas", {"mascota": "Toby"})
        print("Prompt:", prompt.messages[0].content.text)
        r = await a.cliente.call_tool("redactar_aviso", {"mascota": "Toby"})
        print("Sampling:", r.content[0].text)

await demostracion()

## 4. Los documentos: todos o a demanda

In [ ]:
PREGUNTAS = [
    ("cancelar_tarde", "Una clienta cancela hoy la cita que tiene mañana. ¿Le cobramos algo?",
     lambda r: dice(r, r"\b15\b")),
    ("ayuno_gato", "¿Cuántas horas tiene que estar sin comer un gato antes de una anestesia?",
     lambda r: dice(r, r"\b12\b")),
    ("ayuno_conejo", "Un conejo se opera mañana. ¿Cuántas horas de ayuno necesita?",
     lambda r: dice(r, r"\b1 hora", "una hora")),
    ("recargo", "¿Qué recargo tiene una urgencia fuera de horario?", lambda r: dice(r, r"\b60\b")),
    ("cachorros", "¿A cuántas semanas se pone la segunda dosis de la polivalente a un cachorro?",
     lambda r: dice(r, r"\b12\b", "doce")),
    ("desparasitar", "¿Cada cuánto se desparasita por dentro a un perro adulto?",
     lambda r: dice(r, r"\b3 meses", "tres meses")),
    ("cirugias", "¿Cómo se paga una cirugía?", lambda r: dice(r, "mitad")),
    ("collar", "¿Cuántos días tiene que llevar el collar isabelino un perro castrado?",
     lambda r: dice(r, r"\b10\b", "diez")),
]


FONDOS = {"8 documentos": DOCUMENTOS, "21 documentos": {**DOCUMENTOS, **MAS_DOCUMENTOS}}


async def medir_documentos():
    resultados = {}
    variantes = [("ninguno", "8 documentos")] + [(d, f) for f in FONDOS for d in ("todos", "a demanda")]
    for documentos, fondo in variantes:
        filas = []
        async with Anfitrion(crear_servidor(documentos=FONDOS[fondo]), documentos) as anfitrion:
            for clave, pregunta, comprobar in PREGUNTAS:
                respuesta, traza = await agente_mcp(pregunta, anfitrion)
                filas.append({"tarea": clave, "bien": bool(comprobar(respuesta)), "respuesta": respuesta,
                              "tokens_entrada": sum(p["tokens_entrada"] for p in traza),
                              "llamadas": [(l["name"], l["arguments"]) for p in traza for l in p["llamadas"]]})
        nombre = "ninguno" if documentos == "ninguno" else f"{documentos}, {fondo}"
        resultados[nombre] = {"bien": sum(f["bien"] for f in filas),
                              "tokens_entrada": sum(f["tokens_entrada"] for f in filas), "filas": filas}
        print(f"documentos {nombre:24s}", resultados[nombre]["bien"], "de", len(PREGUNTAS),
              resultados[nombre]["tokens_entrada"], "tokens", flush=True)
    return resultados

In [ ]:
documentos = await medir_documentos()

## 5. Reservas a las que les falta un dato

In [ ]:
INCOMPLETAS = [
    ("sin_hora", "Reserva una revisión para Lola con la Dra. Ana Suárez el lunes 26.",
     {"mascota": "Lola", "veterinario": "Ana Suárez", "fecha": "2026-10-26", "hora": "11:00", "motivo": "revisión"}),
    ("sin_veterinario", "Reserva una vacunación para Max el lunes 26 a las 12:00.",
     {"mascota": "Max", "veterinario": "Lucía Fano", "fecha": "2026-10-26", "hora": "12:00", "motivo": "vacunación"}),
    ("sin_fecha", "Reserva una revisión para Toby con el Dr. Pablo Menéndez a las 10:00.",
     {"mascota": "Toby", "veterinario": "Pablo Menéndez", "fecha": "2026-10-27", "hora": "10:00", "motivo": "revisión"}),
    ("sin_motivo", "Reserva una cita para Rocky con la Dra. Lucía Fano el martes 27 a las 16:30.",
     {"mascota": "Rocky", "veterinario": "Lucía Fano", "fecha": "2026-10-27", "hora": "16:30", "motivo": "control de peso"}),
    ("por_la_tarde", "Reserva para Coco una revisión con la Dra. Ana Suárez el lunes 26 por la tarde.",
     {"mascota": "Coco", "veterinario": "Ana Suárez", "fecha": "2026-10-26", "hora": "17:30", "motivo": "revisión"}),
    ("completa", "Reserva para Kira una consulta con el Dr. Pablo Menéndez el lunes 26 a las 9:30.",
     {"mascota": "Kira", "veterinario": "Pablo Menéndez", "fecha": "2026-10-26", "hora": "09:30", "motivo": "consulta"}),
]


def citas_nuevas():
    return [c for c in servidor_clinica.CITAS if c["id"] not in {x["id"] for x in servidor_clinica.CITAS0}]


def es_la_cita(cita, quiere):
    """¿Es la cita la que quería el usuario? El motivo basta con que lo contenga."""
    return (servidor_clinica.MASCOTAS[cita["mascota"]]["nombre"] == quiere["mascota"]
            and cita["veterinario"] == servidor_clinica.veterinario_de(quiere["veterinario"])
            and (cita["fecha"], cita["hora"]) == (quiere["fecha"], quiere["hora"])
            and servidor_clinica.sin_tildes(quiere["motivo"]) in servidor_clinica.sin_tildes(cita["motivo"]))


async def medir_preguntas():
    resultados = {}
    for preguntar in ("no", "falta", "confirmar"):
        filas = []
        for clave, tarea, quiere in INCOMPLETAS:
            servidor_clinica.reiniciar()
            async with Anfitrion(crear_servidor(preguntar), usuario=quiere) as anfitrion:
                respuesta, traza = await agente_mcp(tarea, anfitrion)
                preguntas = anfitrion.preguntas
            nuevas = citas_nuevas()
            filas.append({"tarea": clave, "bien": len(nuevas) == 1 and es_la_cita(nuevas[0], quiere),
                          "inventadas": sum(not es_la_cita(c, quiere) for c in nuevas),
                          "respuesta_coincide": len(nuevas) == 1 and dice(respuesta, nuevas[0]["hora"].lstrip("0"))
                          and dice(respuesta, rf"\b{int(nuevas[0]['fecha'][-2:])}\b"),
                          "respuesta": respuesta, "preguntas": preguntas, "citas_nuevas": nuevas,
                          "llamadas": [(l["name"], l["arguments"]) for p in traza for l in p["llamadas"]]})
        resultados[preguntar] = {"bien": sum(f["bien"] for f in filas),
                                 "citas_con_datos_inventados": sum(f["inventadas"] for f in filas),
                                 "respuestas_que_coinciden": sum(f["respuesta_coincide"] for f in filas),
                                 "preguntas": sum(len(f["preguntas"]) for f in filas), "filas": filas}
        print(f"reservas {preguntar:10s}", {k: v for k, v in resultados[preguntar].items() if k != "filas"}, flush=True)
        print("    " + " ".join(("+" if f["bien"] else "-") + f["tarea"] for f in filas), flush=True)
    return resultados

In [ ]:
reservas = await medir_preguntas()

## Ejercicios

1. Explica con tus palabras quién decide usar cada una de las cuatro cosas del capítulo: una herramienta, un recurso, un prompt y una petición de sampling.

2. Lee con `read_resource` los 21 documentos y cuenta sus tokens. ¿A partir de cuántos documentos, de ese tamaño, compensa leerlos a demanda?

3. Cambia la descripción de `leer_documento` para que dé solo las direcciones, sin los títulos. ¿Encuentra el modelo el documento que necesita?

4. Escribe dos preguntas cuya respuesta esté en dos documentos a la vez. ¿Los lee el modelo los dos?

5. En la reserva sin hora, con la versión que pregunta lo que falta, el modelo eligió una hora sin preguntar aunque la herramienta pedía dejarla vacía. Prueba a decirlo también en el mensaje de sistema. ¿Deja ahora la hora vacía?

6. Haz que el usuario simulado rechace la confirmación de una reserva. ¿Qué responde el agente?

7. Escribe un prompt del servidor para el resumen de la semana de un veterinario y úsalo como primer mensaje del agente. ¿Qué herramientas llama?

8. `redactar_aviso` pide el texto al modelo del anfitrión. Cambia el anfitrión para que, antes de responder, enseñe la petición al usuario y le deje cancelarla. ¿Por qué querría un anfitrión hacerlo?

9. ¿Cuántos tokens de entrada añade al modelo cada pregunta de confirmación? Cuenta también cuántas preguntas hace al usuario cada versión en las seis reservas. ¿Compensa preguntar siempre o solo cuando falta algo?

10. **Proyecto.** Añade a tu proyecto del capítulo 4 dos o tres documentos como recursos, y añade a una de sus herramientas que actúan una confirmación con elicitation. Mide el acierto y los tokens con y sin cada cambio.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Especificación de MCP](https://modelcontextprotocol.io/specification): recursos, prompts, sampling y elicitation. [SDK de Python](https://github.com/modelcontextprotocol/python-sdk). Lecciones 13.10, 13.11 y 13.12 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La Clínica Veterinaria Naranco, sus datos y sus documentos son ficticios. Código y experimentos propios.